In [1]:
!pip install gradio transformers torch reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.0 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import sqlite3
import re
from datetime import datetime

In [ ]:
schema = {
    "users": ["id", "name", "email", "signup_date", "age", "country", "status"],
    "orders": ["id", "user_id", "product_name", "amount", "order_date", "status"],
    "products": ["id", "name", "price", "category", "stock"],
    "transactions": ["id", "user_id", "amount", "date", "type"]
}

In [ ]:
def is_safe(query):
    dangerous_keywords = ["DROP", "DELETE", "ALTER", "TRUNCATE", "INSERT", "UPDATE"]

    for word in dangerous_keywords:
        if word.lower() in query.lower():
            return False
    return True

In [ ]:
def get_table_name(user_input):
    if "user" in user_input:
        return "users"
    elif "order" in user_input:
        return "orders"
    elif "product" in user_input:
        return "products"
    elif "transaction" in user_input:
        return "transactions"
    else:
        return "users"

In [ ]:
def extract_conditions(text):
    condition = ""

    amount_match = re.search(r'greater than (\d+)', text)
    if amount_match:
        condition = f"amount > {amount_match.group(1)}"

    if "last week" in text:
        condition += " AND DATE(date) >= DATE('now','-7 days')"

    if "last month" in text:
        condition += " AND DATE(signup_date) >= DATE('now','-1 month')"

    return condition

In [ ]:
def generate_sql(user_input):

    if not is_safe(user_input):
        return "Unsafe query detected!"

    table = get_table_name(user_input)
    condition = extract_conditions(user_input)

    sql = f"SELECT * FROM {table}"

    if condition:
        sql += f" WHERE {condition}"

    sql += " LIMIT 10"

    return sql

In [ ]:
def chatbot(query):
    sql_query = generate_sql(query)
    return sql_query

interface = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="SQL Query Generator",
    description="Convert natural language into SQL query"
)

interface.launch()